In [ ]:
import sys
import warnings
from functools import partial

import numpy as np
import torch
import yaml
from sklearn.metrics import mean_squared_error
from tabpfn import TabPFNRegressor

sys.path.append("..")
from src.data_generation.data_preperation import data_preparation
from src.model.finetune import finetune
cfg = yaml.safe_load(open("../config.yaml"))

warnings.filterwarnings("ignore", message="Running on CPU with more than")

In [ ]:
N_CONTEXT = 20
data_provider = partial(data_preparation, cfg, n_context=N_CONTEXT)

In [ ]:
RUN_NAME = "ssvi_fixed_context_20"
finetune(data_provider, run_name=RUN_NAME, n_epochs=100, n_surfaces_per_epoch=20, n_val_surfaces=10)

In [ ]:
N_TEST_SURFACES = 50
N_ESTIMATORS = 1 # finetuned with 1
test_train, test_test = data_preparation(cfg, N_TEST_SURFACES, N_CONTEXT)

In [ ]:
baseline = TabPFNRegressor(
    n_estimators=N_ESTIMATORS, inference_config={"FINGERPRINT_FEATURE": False},
)

finetuned = TabPFNRegressor(
    fit_mode="fit_preprocessors", n_estimators=N_ESTIMATORS,
    inference_config={"FINGERPRINT_FEATURE": False},
)
finetuned._initialize_model_variables()
finetuned_state = torch.load(f"../checkpoints/{RUN_NAME}/best.pt", map_location="cpu")
finetuned.model_.load_state_dict(finetuned_state)

In [ ]:
def eval_surfaces(model, train_list, test_list, reload_state=None):
    rmses, maes, mapes, mspes = [], [], [], []
    for (X_tr, y_tr), (X_te, y_te) in zip(train_list, test_list):
        model.fit(X_tr, y_tr)
        if reload_state is not None:
            model.model_.load_state_dict(reload_state)
        y_pred = model.predict(X_te)
        rmses.append(np.sqrt(mean_squared_error(y_te, y_pred)))
        maes.append(np.mean(np.abs(y_te - y_pred)))
        mapes.append(np.mean(np.abs((y_te - y_pred) / y_te)) * 100)
        mspes.append(np.mean(((y_te - y_pred) / y_te) ** 2) * 100)
    return np.mean(rmses), np.mean(maes), np.mean(mapes), np.mean(mspes)

In [ ]:
print("Evaluating baseline")
b_rmse, b_mae, b_mape, b_mspe = eval_surfaces(baseline, test_train, test_test)
print("Evaluating finetuned")
f_rmse, f_mae, f_mape, f_mspe = eval_surfaces(finetuned, test_train, test_test, reload_state=finetuned_state)

print(f"\n{'':20s} {'Baseline':>12s} {'Finetuned':>12s} {'Delta':>10s}")
print("-" * 56)
for name, b, f in [("RMSE", b_rmse, f_rmse), ("MSE (%)", b_mspe, f_mspe), ("MAE", b_mae, f_mae), ("MAPE (%)", b_mape, f_mape)]:
    print(f"{name:20s} {b:12.4f} {f:12.4f} {(f-b)/b * 100:+10.2f}%")